# **Probe and activation patching: Practice**

In the lesson “Illustrative examples of MI”, the Othello-GPT case separated two things: the board representation is *decodable* from the hidden states (a probe showed this) and the model *uses* it (an intervention showed this). In the lesson “Methods of mechanistic interpretability”, the same pair appears on the method map as reading and intervention. Here both methods are applied to one task and one model, and their answers are compared layer by layer.

The model is GPT-2 small from TransformerLens (12 layers, residual stream of dimension 768); the task is IOI from case 3: in the prompt “When Mary and John went to the store, John gave a drink to” the correct continuation is “ Mary” (the indirect object, IO), and the repeated name “ John” is the subject (S).

In this practice you will:

* measure how confidently the model solves IOI on 200 prompts: the difference between the IO and S logits;
* train a linear probe on the residual stream of every layer at two positions — the second mention of the subject (S2) and the last token (END) — and find from which layer the probe tells which name in the prompt is the IO;
* run activation patching on the same activations and find at which layers and positions they are causally needed for the answer;
* compare the two curves and find the layers where the information is decodable but the model does not take it from there.

The tasks are numbered; their answers are entered in the lesson “Probe and activation patching: Practice”.

## **Environment setup**
A CPU is enough: the model is small, and all computations here are forward passes and logistic regression.

In [ ]:
!pip install transformer-lens==3.9.0 scikit-learn -q

In [ ]:
import random
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from transformer_lens import HookedTransformer

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_grad_enabled(False)  # only forward passes here
model = HookedTransformer.from_pretrained("gpt2-small", device="cpu")
print(model.cfg.n_layers, model.cfg.d_model)

## **IOI prompts**

We build 200 prompts from one template. Names, places, and objects are single-token words, so token positions coincide across all prompts: S2 is position 10, END is the last position 14 (position 0 is the special token `<|endoftext|>`).

In half of the prompts the IO is the first of the two names (“When Mary and John …, John gave … to”), in the other half the second (“When Mary and John …, Mary gave … to”). The label `y` = 1 if the IO is the first name. This is what the probe will predict.

For activation patching, every prompt needs a corrupted partner: the same prompt with the other name repeated. In it, IO and S swap roles, and the correct answer switches to the other name.

In [ ]:
NAMES = [" Mary", " John", " Alice", " Bob", " Sarah", " Tom", " Anna", " James", " Emma", " David",
         " Kate", " Paul", " Laura", " Mark", " Lisa", " Peter", " Jane", " Mike", " Susan", " Steve"]
PLACES = [" store", " park", " school", " garden", " station", " office"]
OBJECTS = [" drink", " book", " ring", " letter", " ball", " gift"]
TEMPLATE = "When{A} and{B} went to the{P},{S} gave a{O} to"
N = 200

rows = []
for i in range(N):
    a, b = random.sample(NAMES, 2)
    io_first = i % 2 == 0  # in half of the prompts the IO is the first name
    rows.append(dict(A=a, B=b, P=random.choice(PLACES), O=random.choice(OBJECTS),
                     S=b if io_first else a, IO=a if io_first else b, y=int(io_first)))

clean_tokens = model.to_tokens([TEMPLATE.format(**r) for r in rows])
corr_tokens = model.to_tokens([TEMPLATE.format(**{**r, "S": r["IO"]}) for r in rows])  # corrupted partner: the other name is repeated
io_tok = torch.tensor([model.to_single_token(r["IO"]) for r in rows])
s_tok = torch.tensor([model.to_single_token(r["S"]) for r in rows])
y = np.array([r["y"] for r in rows])
S2, END = 10, clean_tokens.shape[1] - 1
print(model.to_str_tokens(clean_tokens[0]))
print(model.to_str_tokens(corr_tokens[0]))
print("S2:", model.to_str_tokens(clean_tokens[0])[S2], " END:", model.to_str_tokens(clean_tokens[0])[END])

## **How confidently the model solves IOI**

The measure is the logit difference at the last position: $\mathrm{LD} = \mathrm{logit}(\mathrm{IO}) - \mathrm{logit}(\mathrm{S})$, where IO and S are taken from the *clean* prompt. On clean prompts it is positive if the model chooses the IO; on corrupted prompts it is negative, because there the correct answer is the S of the clean prompt.

Complete the function `logit_diff`.

In [ ]:
def logit_diff(logits):
    last = logits[:, -1]
    return ...  # Your code here


clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corr_logits = model(corr_tokens)
ld_clean, ld_corr = logit_diff(clean_logits), logit_diff(corr_logits)
print("mean LD clean:", round(ld_clean.mean().item(), 2), " share LD > 0:", (ld_clean > 0).float().mean().item())
print("mean LD corrupted:", round(ld_corr.mean().item(), 2))

**Task 1. What is the mean logit difference between IO and S over the 200 clean prompts? Round to two decimal places.**

## **A linear probe by layer**

The probe is a logistic regression on the residual stream activations at one position of one layer. Layer $l$ here is the input of block $l$ (`blocks.l.hook_resid_pre`); layer 12 is the output of the last block. The probe is trained on 120 prompts and tested on the remaining 80; an accuracy of 0.5 is chance level.

Complete the probe training and the accuracy on the held-out prompts.

In [ ]:
L = model.cfg.n_layers
perm = np.random.permutation(N)
train, test = perm[:120], perm[120:]


def resid(layer, pos):
    name = f"blocks.{layer}.hook_resid_pre" if layer < L else f"blocks.{L - 1}.hook_resid_post"
    return clean_cache[name][:, pos].numpy()


probe_acc = {}
for pos_name, pos in [("S2", S2), ("END", END)]:
    accs = []
    for layer in range(L + 1):
        h = resid(layer, pos)
        accs.append(...)  # Your code here
    probe_acc[pos_name] = accs
    print(pos_name, [round(a, 3) for a in accs])

**Task 2. From which layer does the probe at the END position first separate the label `y` with accuracy 1.0? Write the layer number.**

**Task 3. From which layer does the probe at the S2 position separate the label `y` with accuracy 1.0? Write the layer number.**

## **Activation patching by layer**

We run the model on the corrupted prompt and, at one position of one layer, substitute the residual stream activation from the clean prompt. If this activation carries what the model uses for the answer, the logit difference returns to the clean one. The restored fraction

$$r_l = \frac{\overline{\mathrm{LD}}_{\text{patched}} - \overline{\mathrm{LD}}_{\text{corr}}}{\overline{\mathrm{LD}}_{\text{clean}} - \overline{\mathrm{LD}}_{\text{corr}}},$$

where the bar is the mean over the 200 prompts: $r_l = 1$ means the substitution fully restored the answer, $r_l = 0$ means it changed nothing.

Complete the hook that substitutes the activation, and the formula for $r_l$.

In [ ]:
patch_frac = {}
for pos_name, pos in [("S2", S2), ("END", END)]:
    fracs = []
    for layer in range(L):
        def patch(act, hook):
            ...  # Your code here
            return act

        patched = model.run_with_hooks(corr_tokens, fwd_hooks=[(f"blocks.{layer}.hook_resid_pre", patch)])
        ld_patched = logit_diff(patched)
        fracs.append(...)  # Your code here
    patch_frac[pos_name] = fracs
    print(pos_name, [round(f, 3) for f in fracs])

**Task 4. From which layer does patching at the S2 position stop restoring the answer: $r_l$ is below 0.1 for the first time? Write the layer number.**

**Task 5. What is the restored fraction $r_l$ for patching at the END position at the layer you found in task 2? Round to two decimal places.**

## **Comparison: decodable versus used**

Let us put both curves in one table. Where the probe gives 1.0 and patching gives about zero, the information is present in the activation, but the model does not take it from here for the answer.

In [ ]:
print("layer  probe S2  patch S2  probe END  patch END")
for layer in range(L + 1):
    ps = f"{patch_frac['S2'][layer]:8.3f}" if layer < L else "       —"
    pe = f"{patch_frac['END'][layer]:9.3f}" if layer < L else "        —"
    print(f"{layer:5d}  {probe_acc['S2'][layer]:8.3f}  {ps}  {probe_acc['END'][layer]:9.3f}  {pe}")

**Task 6. At the S2 position, at layers 9–12, the probe separates the label with accuracy 1.0, while patching restores almost nothing. What follows from this?**

## **Conclusion**

- A probe answers the question of whether the information *is present* in the activation: at the S2 position, the label `y` is decodable from the first layer on and stays decodable up to the model output.
- Activation patching answers the question of whether the model *uses* this activation: at the S2 position the substitution restores the answer only at the early and middle layers, and at the END position only from the late layers, once the name for the answer has been moved there.
- The layers where the probe gives 1.0 and patching gives about zero are the gap between decodability and causal role that the method map talks about: high probe accuracy alone does not prove that the model uses the representation. That is why the Othello-GPT case tested the board representation with an intervention.
- What exactly moves the information from the S2 position to the END position is shown by the analysis of the IOI circuit by attention heads (Wang et al., 2023): these are the topics of the lessons of the module “Mechanistic interpretability: circuits and reading representations”, starting with the lesson “What is a circuit”.